# Method B: MentalRoBERTa on Colab

Runtime > Change runtime type > **T4 GPU**. Before running:
1. Accept the model terms at https://huggingface.co/mental/mental-roberta-base
2. In Colab, open the key icon (Secrets) and add two secrets, both with notebook access on: `HF_TOKEN` (Hugging Face read token) and `KAGGLE_API_TOKEN` (your Kaggle token).

Results and predictions are saved to Google Drive after every run, so a disconnect does not lose finished runs.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - change the runtime type')

In [ ]:
!git clone https://github.com/Abhinavxox/reddit-mental-health-timing.git
%cd reddit-mental-health-timing
!pip install -q -r requirements.txt kaggle transformers

In [ ]:
!mkdir -p phase1/data/raw
!kaggle datasets download -d kamaruladha/mental-disorders-identification-reddit-nlp -p phase1/data/raw --unzip

In [ ]:
# Replicate samples and splits (same code and seeds as the laptop runs, so the rows are identical).
# Check the number first: use the same N and REPS as phase1/run_final.sh.
N = 60000
REPS = [0, 1, 2]
for k in REPS:
    !python phase1/src/prepare.py --source kaggle --n {N} --rep {k}
    !python phase1/src/splits.py --source kaggle --rep {k}

In [ ]:
# Smoke test first: ONE run (text only, temporal_late, replicate 0), about 15 to 20 minutes on a T4.
# Read the printed line before running the rest.
!cd phase1 && python run.py --source kaggle --model roberta --features text --split temporal_late --reps 0

In [ ]:
# All runs: reps x {random_matched, temporal_late} x {text, text_time}. Copies results to Drive after each rep.
import shutil
for k in REPS:
    !cd phase1 && python run.py --source kaggle --model roberta --all --reps {k}
    shutil.copytree('phase1/results/kaggle/final', f'/content/drive/MyDrive/timing_matters_B/rep{k}', dirs_exist_ok=True)

Download `results_roberta.csv` and the `preds/` folder from Drive, put them into `phase1/results/kaggle/final/` in the repo, and run `python phase1/src/stats.py`.